<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor: experiment notebook

This notebook compares five formulations for the periodic two-well **committor probability** PDE, following the methods in [Tepakbong et al. (2026)](https://arxiv.org/abs/2607.19167).

**Workflow:** set `DEVICE` below, choose the matching Colab runtime once before Setup (`"cuda"` → GPU, `"cpu"` → CPU), then run Setup, the CPU preflight, the small code check, CPU preparation, and training in that same runtime. Setup runs tests with CUDA hidden in a child process; reference and Monte Carlo preparation use CPU even when the notebook has a GPU. No runtime switch is needed between preparation and training. The longer preparation reports progress and saves to one dated Drive folder.

[Experiment guide: full derivation, loss definitions, saved-file map, and references](https://github.com/n2pakbong/boundary-adapted-pinns/blob/main/experiments/committor/periodic_two_well/README.md).

## 1. The mathematical problem

Consider overdamped Langevin dynamics on the flat torus $\mathbb T^2=[0,1)^2$:

$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t \pmod 1,
\qquad
V(x_1,x_2)=a\left[1-\cos\!\left(4\pi(x_1-\tfrac14)\right)\right]
+b\left[1-\cos(2\pi x_2)\right].
$$

For the default settings, $A$ and $B$ are disjoint periodic disks, centered at $(1/4,0)$ and $(3/4,0)$. The **forward committor** is

$$
q(x)=\mathbb P_x(\tau_B<\tau_A),
$$

the probability that a trajectory started at $x$ reaches $B$ before $A$. It solves

$$
\mathcal L q
=-\nabla V\cdot\nabla q+\varepsilon\Delta q=0
\quad\text{in }D=\mathbb T^2\setminus(A\cup B),
$$

with $q=0$ on $\partial A$ and $q=1$ on $\partial B$. Thus this notebook studies a hitting probability, not a mean escape time.

The starting values are $a=1$, $b=0.6$, $\varepsilon=0.4$, and periodic chordal radius $r=0.5$. The finite-difference solution is a numerical target for error measurement, not exact ground truth.

This notebook supports one problem family: a flat unit torus with two equal-radius periodic disks and the cosine potential shown above. You can change the potential amplitudes, diffusion strength, disk centers, and common radius below. The torus, disk shape, potential formula, and cosine phase centers stay fixed. Moving a disk does not move a potential well. The runner rejects disks that overlap or touch. A different domain, disk geometry, or potential formula requires code changes to the problem implementation and its samplers.

### Change the potential and the two boundary sets

The domain stays the flat unit torus: each coordinate wraps around at 0 and 1. For readability, enter each center coordinate in the range from 0 up to (but not including) 1; for example, 0.25 is one quarter of the way around the torus. The coordinates wrap periodically. Set A is the source, where the committor is 0. Set B is the target, where it is 1. Both use the same periodic chordal radius. If the two closed disks overlap or touch, the run stops with an error. The radius must lie strictly between 0 and 1.

`POTENTIAL_A` and `POTENTIAL_B` are the amplitudes in the displayed cosine formula; `DIFFUSION_EPSILON` is the noise strength and must be positive. Moving a disk center does not move a cosine well. This interface cannot change the dimension, domain, disk shape, or potential formula.


In [ ]:
# Edit these ordinary Python values to change the example problem.
POTENTIAL_A = 1.0
POTENTIAL_B = 0.6
DIFFUSION_EPSILON = 0.4
SOURCE_CENTER = (0.25, 0.0)
TARGET_CENTER = (0.75, 0.0)
COMMON_CHORDAL_RADIUS = 0.5


## The network

The model is an ordinary PyTorch module. It maps each coordinate pair to four periodic features, then uses a fully connected MLP with four 64-unit hidden layers, tanh activations, and one scalar output. The sine/cosine input map makes the output periodic in both coordinates. The paper does not report a periodic committor model; its mean-escape-time experiments use different MLPs (width 30, tanh, with two hidden layers for OU and three for the double well). This model is the notebook's explicit choice for this added torus example. The strong-form PDE loss differentiates twice, so choose an activation that behaves suitably under second derivatives.

Change the layers and activation directly in the class below. `nn.Linear` uses PyTorch's default initialization; change it with ordinary `torch.nn.init` code if needed. Keep a periodic input map or use another periodic architecture if the model is still meant to live on the torus.

In [ ]:
import torch
from torch import nn

class PeriodicCommittorNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(4, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x has shape [n, 2]; encode both unit-periodic coordinates.
        phase = 2.0 * torch.pi * x
        features = torch.cat((torch.sin(phase), torch.cos(phase)), dim=-1)
        return self.layers(features).squeeze(-1)

print(PeriodicCommittorNetwork())

### Interior sampling

This function proposes points uniformly from `[0, 1)²` and rejects points inside A or B, so accepted points are uniform by area on the punctured torus D. It supplies both PDE collocation points and Monte Carlo anchors. Edit the function to use another distribution; every point must remain in D. A nonuniform law changes the empirical averages unless you add importance weights. Boundary points use a separate sampler: uniform radial angles with arclength weights in boundary losses and Nitsche integrals.

In [ ]:
def sample_training_points(problem, n, *, device, dtype, generator=None):
    # Uniform rejection sampling on D = [0, 1)^2 \\ (A union B).
    accepted = []
    remaining = n
    for _ in range(100):
        proposal = torch.rand(
            max(2 * remaining, 32), 2,
            device=device, dtype=dtype, generator=generator,
        )
        inside_D = problem.domain_mask(proposal)
        take = proposal[inside_D][:remaining]
        if len(take):
            accepted.append(take)
            remaining -= len(take)
        if remaining == 0:
            return torch.cat(accepted, dim=0)
    raise RuntimeError("Could not sample enough points in D; check the geometry.")

## 2. What runs, and what uses compute

| Stage | Work | Compute |
|---|---|---|
| Setup | Install code and run tests in a child process with CUDA hidden | CPU |
| Preflight | Tiny one-epoch run of all five methods | CPU |
| Preparation | Finite-difference references and shared Monte Carlo labels | CPU, even in a GPU runtime; saved to Drive |
| Training | Selected networks and solution-error evaluation | The configured device |
| Inspect/evaluate | Saved curves, losses, errors; checkpoint evaluation | CPU |

Set `DEVICE` to `"cuda"` and choose a GPU runtime before Setup for GPU training, or set it to `"cpu"` and choose CPU. Keep that runtime through preparation and training: the preparation stage uses CPU either way, and switching later would reset the notebook's live Python state. Edit the ordinary Python settings and model/sampling functions in this notebook; the JSON pilot file is for the separate command-line workflow. This single-seed comparison is exploratory, not a final paper result.

### Pick the methods to compare

Edit the Python list in the next cell. It accepts these names: `simple_pinn`, `boundary_adapted`, `over_enforced`, `under_enforced`, and `variational_nitsche`. Their trial functions and losses are described above. The run trains only the methods in this list; the small preflight still checks all five methods.

In [ ]:
METHODS_TO_RUN = [
    "simple_pinn",
    "boundary_adapted",
    "over_enforced",
    "under_enforced",
    "variational_nitsche",
]


### Choose hardware, precision, and random seed

These are ordinary Python values. CPU preparation always stays on CPU. `DEVICE` chooses where neural-network training runs. The seed controls the model initialization and random training points.

In [ ]:
DEVICE = "cuda"       # Use "cpu" or "cuda" (or "cuda:0").
DTYPE = "float64"     # Use "float32" or "float64".
SEED = 7


## 3. Setup and CPU tests

First set `DEVICE` in section 2 and choose the matching Colab runtime: GPU for `"cuda"`, CPU for `"cpu"`. Then run Setup. It connects Google Drive, installs the package into this notebook's Python kernel, and runs the tests in a separate process with CUDA hidden, so a GPU runtime stays attached and available for training. Setup does not train. Preparation and results are saved to Drive.

Keep this same runtime through preflight, preparation, and training. If you manually change runtimes, Colab resets the live Python session; run Setup and the notebook's code cells again, but do not repeat a completed preparation saved on Drive.

In [ ]:
# CPU setup: select a Git branch, install it, and run mathematical and tiny training tests.
from pathlib import Path
import base64
import os
import subprocess
import sys

from google.colab import userdata, drive

# Connect Drive before setup/tests so mount problems appear at the start.
def ensure_drive_mounted():
    if Path('/content/drive/MyDrive').is_dir():
        return
    try:
        drive.mount('/content/drive', force_remount=True)
    except ValueError as error:
        raise RuntimeError(
            'Google Drive could not be mounted during Setup. No experiment folder or preparation was created. '
            'Check that Colab is signed into the Google account for your Drive. '
            'Colab also documents failures when My Drive has roughly 10,000 or more items directly in its root. '
            'If the Drive looks accessible, reconnect to the Colab runtime and rerun Setup.'
        ) from error

ensure_drive_mounted()

CPU_PREFLIGHT_PASSED = False

def run_logged(command, log_path, append=False):
    """Show output and offer the complete log as a download if the command fails."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a" if append else "w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        print(f"\nCommand failed (exit {return_code}). The full traceback is above.")
        try:
            from google.colab import files
            files.download(str(log_path))
            print("Attach the downloaded console.log if you need help diagnosing the error.")
        except Exception as download_error:
            print(f"Log download did not start: {download_error}. Copy the traceback above.")
        raise RuntimeError("Command failed. The traceback and downloadable log are above.")

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = input('Git branch to install [main]: ').strip() or 'main'

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not a Git clone. Rename that temporary folder, then rerun setup.')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    # Publicly readable repositories can be cloned without authentication.
    pass

git_auth = ['git']
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_auth += ['-c', f'http.extraheader=AUTHORIZATION: basic {auth}']

try:
    if not REPO.exists():
        subprocess.run([
            *git_auth, 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run([
            *git_auth, '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH
        ], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not fetch the selected branch. Check the branch name and repository access, then rerun Setup.'
    ) from error
finally:
    del token
    if 'auth' in globals():
        del auth

%cd /content/boundary-adapted-pinns
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
import importlib
importlib.invalidate_caches()
try:
    import boundary_pinn
except ModuleNotFoundError as error:
    raise RuntimeError(
        "The package did not install into this notebook's Python kernel. "
        "Check the installation output above, then rerun Setup."
    ) from error
print('Installed boundary_pinn into this notebook kernel.')
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import torch
# Run tests with CUDA hidden in this child process. The notebook kernel keeps its GPU.
test_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "PYTHONUNBUFFERED": "1"}
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True, env=test_env)
print('Setup and CPU tests passed. The notebook runtime remains unchanged. Code revision:', revision)


## 4. CPU preflight: test the packaged runner

Run this after Setup passes. It runs all five methods for one epoch using the repository's tiny preset, default model and samplers, one fixed Monte Carlo anchor, eight short paths, and coarse finite-difference grids. This checks that the installed runner works; it does not run the model or sampler code you edit in this notebook. Section 11 runs a small smoke check of your model, sampler, optimizer, and scheduler before the longer CPU preparation.

Outputs go under `/content/periodic_committor_preflight`, temporary Colab storage. The cell sets a flag only after it completes; preparation and training require this check to pass.

In [ ]:
# CPU only: run the packaged preset in temporary Colab storage.
from pathlib import Path
import shutil

CPU_PREFLIGHT_PASSED = False
preflight_dir = Path('/content/periodic_committor_preflight')
if preflight_dir.exists():
    shutil.rmtree(preflight_dir)
preflight_dir.mkdir()
PREFLIGHT_ROOT = str(preflight_dir)
print('CPU preflight outputs (temporary Colab storage):', PREFLIGHT_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], preflight_dir / 'console.log')
if not (preflight_dir / 'summary.json').is_file():
    raise RuntimeError('CPU preflight produced no summary.json; check the command output above.')
CPU_PREFLIGHT_PASSED = True
print('CPU preflight passed. The full CPU experiment preparation is now available.')

## 5. Boundary conditions and the five trial functions

For each boundary set $j\in\{A,B\}$, the code builds a smooth level-set function $\omega_j$ and a normalized distance-like function $d_j$:

$$
\omega_j(x)=4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)-r^2,
\qquad
d_j(x)=\frac{\omega_j(x)}
{\sqrt{\omega_j(x)^2+\lvert\nabla\omega_j(x)\rvert^2}}.
$$

Here $c_j$ is the center of set $j$. The boundary is where $\omega_j=0$; in the exterior domain, both $d_A$ and $d_B$ are positive. Define the smooth lift $g$ and the two-boundary factor $\rho$ by

$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},
\qquad
\rho(x)=\frac{d_A(x)d_B(x)}
{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$

On $\omega_j=0$, where $\nabla\omega_j\ne0$, differentiation gives $\nabla d_j=\nabla\omega_j/|\nabla\omega_j|$, hence $|\nabla d_j|=1$. This is the first-order normalization: near its own boundary, $d_j$ changes at unit rate in the normal direction.

To see why the two-set factor is also normalized, write $F(a,b)=ab/\sqrt{a^2+b^2}$, so $\rho=F(d_A,d_B)$. On $\partial A$, the sets are disjoint and $d_B>0$. Since $\partial_aF(0,d_B)=1$ and $\partial_bF(0,d_B)=0$, we get $\nabla\rho=\nabla d_A$ and $|\nabla\rho|=1$. The same calculation gives $\nabla\rho=\nabla d_B$ and unit gradient on $\partial B$. The outward normal of the exterior domain points into a removed set, so $\partial_{n_D}\rho=-1$ on either boundary.

A plain product $d_A d_B$ also vanishes on both boundaries, but its normal derivative is scaled by the distance to the other set. The normalized $\rho$ removes that scale. Let $s$ be distance measured into $D$ from either boundary. Since $\partial_{n_D}\rho=-1$, $\rho=s+O(s^2)$ there. Thus $\rho^\alpha N_\theta$ behaves like $s^\alpha N_\theta$: for $\alpha=1$, its normal derivative at the boundary is finite and equals $-N_\theta$; for $\alpha>1$, its first normal derivative vanishes; for $\alpha<1$, it can be singular when $N_\theta$ is nonzero on the boundary. Thus $g=0$ on $\partial A$, $g=1$ on $\partial B$, and $\rho=0$ on both. The three exact-boundary trial functions use

$$
q_\theta(x)=g(x)+\rho(x)^\alpha N_\theta(x).
$$

Boundary-adapted uses $\alpha=1$, over-enforced uses $\alpha=3/2$, and under-enforced uses $\alpha=1/2$. Each satisfies both Dirichlet values by construction. The network output is periodic because the network receives sine and cosine coordinates. For background on the standard PINN residual and exact distance-function boundary enforcement, see [Raissi et al. (2019)](https://doi.org/10.1016/j.jcp.2018.10.045) and [Sukumar and Srivastava (2022)](https://doi.org/10.1016/j.cma.2021.114333); the exponent variants listed here are the specific cases in this experiment.

## 6. Which loss belongs to which method?

Every method uses the same interior collocation points and the same Monte Carlo anchor labels within a comparison. Write the interior residual loss and shared solution-value data loss as

$$
\widehat J_{\rm pde}(\theta)
=\frac{1}{N_D}\sum_{i=1}^{N_D}
\left|\mathcal Lq_\theta(x_i)\right|^2,
\qquad
\widehat J_{\rm data}(\theta)
=\frac{1}{N_{\rm data}}\sum_{i=1}^{N_{\rm data}}
\left(q_\theta(z_i)-\widehat q_i\right)^2.
$$

| Method | Trial function | Trained loss components | Boundary treatment |
|---|---|---|---|
| Standard PINN (`simple_pinn`) | $q_\theta=N_\theta$ | PDE, Monte Carlo data, ordinary boundary residual | Samples both boundaries and penalizes the Dirichlet residual. |
| Boundary-adapted | $g+\rho N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Over-enforced | $g+\rho^{3/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Under-enforced | $g+\rho^{1/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Variational/Nitsche | $q_\theta=N_\theta$ | Interior-plus-flux loss + Nitsche boundary loss + Monte Carlo data | Weakly imposes the boundary values using Nitsche's method. |

Only the standard PINN has the ordinary boundary-residual component:

$$
\widehat J_{\rm bc}
=\frac{1}{N_A}\sum_i q_\theta(x_i^A)^2
+\frac{1}{N_B}\sum_i\bigl(q_\theta(x_i^B)-1\bigr)^2.
$$

The boundary-adapted, over-enforced, and under-enforced methods have **no extra boundary penalty**. The variational method's stabilization term belongs to the Nitsche weak formulation; it is not applied to an exact-boundary ansatz. The adaptive objective and exact component names are defined in Section 8; the variational method is derived in Section 9. The main experiment guide links each equation to its implementation.

## 7. Monte Carlo labels: the slow CPU stage and its live progress

The committor also has a stochastic representation. Starting at anchor $z_i$, independent Euler–Maruyama trajectories approximate the Langevin dynamics:

$$
X_{n+1}
=\left[X_n-\nabla V(X_n)\Delta t
+\sqrt{2\varepsilon\Delta t}\,\xi_n\right]\pmod 1,
\qquad \xi_n\sim\mathcal N(0,I_2).
$$

This carries over the pointwise solution-value regularizer used in the mean-escape-time notebooks: here the target value is the committor probability, and it complements rather than replaces the PDE residual. Each trajectory stops on first entering $A$ or $B$, or when it reaches the configured step cap. The label is the fraction of resolved trajectories that hit $B$ first:

$$
\widehat q_i
=\frac{\#\{\text{resolved trajectories from }z_i\text{ that hit }B\text{ first}\}}
{\#\{\text{resolved trajectories from }z_i\}}.
$$

Capped, unresolved paths are **not** counted as hits of $A$. Training stops with an error if an anchor's resolved fraction is below the configured minimum. The reported binomial standard error is conditional on the resolved paths; if paths remain unresolved, the guide explains the possible censoring bias and time-step error.

**What you will see while it runs:** reports show the step cap, paths resolved to (A) and (B), paths still active, anchors meeting the configured resolution threshold, the minimum and mean resolved fractions, and elapsed time. They appear at the configured fraction of the step cap or after the configured time interval, whichever comes first. The final resolution summary appears before training. After generation, the next cell saves and displays a plot of anchor locations colored by their estimated committor values, plus each label with its estimated Monte Carlo standard error. All methods share the same anchors and labels in `monte_carlo_labels.npz` and `monte_carlo_labels.json`. Neural-network approaches to committor computation are discussed, for example, by [Khoo, Lu, and Ying (2018)](https://arxiv.org/abs/1802.10275) and [Song, Cameron, and Yang (2023)](https://arxiv.org/abs/2306.12268).

### Set the amount and accuracy of Monte Carlo label data

An **anchor** is one input location where the notebook estimates the committor. `N_ANCHORS` is the number of locations; `PATHS_PER_ANCHOR` is the number of independent simulated trajectories per location. More paths reduce sampling noise and take longer.

Each path uses Euler–Maruyama with step size `TIME_STEP` and stops after hitting A or B, or after `MAX_STEPS`; their product is the maximum simulated time. A capped path is unresolved, not counted as a hit of A. If an anchor’s resolved fraction is below `MINIMUM_RESOLVED_FRACTION`, preparation stops with an error rather than silently accepting an unreliable label. `MONTE_CARLO_SEED` controls this simulation separately from network training.

The path count, step cap, and time step must be positive; the minimum resolved fraction must lie between 0 and 1. `PROGRESS_INTERVAL_FRACTION` must be greater than 0 and no more than 1. It is a fraction: the default 0.01 means 1% of the step cap. Progress appears when either that fraction or the wall-clock interval is reached, whichever comes first. The cell shows live progress; `monte_carlo_labels.json` saves the completed run settings and resolution counts.


In [ ]:
# Monte Carlo label-generation settings.
N_ANCHORS = 32
PATHS_PER_ANCHOR = 256
TIME_STEP = 0.001
MAX_STEPS = 100000
MINIMUM_RESOLVED_FRACTION = 0.9
MONTE_CARLO_SEED = 1007
PROGRESS_INTERVAL_FRACTION = 0.01
PROGRESS_INTERVAL_SECONDS = 10


## 8. Adaptive weighting in the paper's notation

The paper denotes the total adaptive objective by $\widehat J$ and its active empirical loss components by $\widehat J_i$. For each component, $\lambda_i>0$ is a trainable scale. Its objective (Eq. (33)) is

$$
\widehat J(\theta,\{\lambda_i\}_{i\in\mathcal I})
=\sum_{i\in\mathcal I}\frac{1}{2\lambda_i^2}\widehat J_i(\theta)
+\sum_{i\in\mathcal I}\log\lambda_i.
$$

See [Tepakbong et al. (2026), Eq. (33)](https://arxiv.org/abs/2607.19167). The active set $\mathcal I$ follows the method being trained:

- Standard PINN (simple_pinn): $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm bc}$ is the sampled Dirichlet residual.
- Boundary-adapted, over-enforced, and under-enforced: $\{\mathrm{pde},\mathrm{data}\}$. Their boundary values are built into the ansatz, so there is no boundary-residual term.
- Variational/Nitsche: $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm pde}$ is the squared interior-plus-flux residual, and $\widehat J_{\rm bc}$ is the squared Nitsche boundary contribution (symmetric consistency plus stabilization). This is not the standard PINN's pointwise boundary residual.

For this committor, $\widehat J_{\rm pde}$ uses the residual $\mathcal Lq_\theta=0$, and $\widehat J_{\rm data}$ compares $q_\theta(z_i)$ with the simulated committor label $\widehat q_i$. The loss definitions are in Section 6 and the Nitsche split is derived in Section 9. In the training logs, the three optimized Nitsche components are pde_loss, boundary_loss, and data_loss. The separate Nitsche term mean squares and the full weak-residual mean square are diagnostics, not separately weighted losses.

**How the code's saved scale corresponds to $\lambda_i$.** The code stores each trainable scale in log form, $s_i=\log\sigma_i$, and multiplies each raw loss by $e^{-2s_i}$. The paper's parameter is $\lambda_i=e^{s_i}/\sqrt{2}$, so $e^{-2s_i}=1/(2\lambda_i^2)$. Thus each weighted component has exactly the paper's coefficient. The code's total objective differs from the paper's displayed objective only by the constant $|\mathcal I|\log(2)/2$, which has zero gradient and does not change training. The raw loss components and effective weights are logged separately. The trainable-scale construction follows [Kendall, Gal, and Cipolla (2018)](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).

The notebook's `training.log_interval_seconds` setting controls the wall-clock interval between live console reports. The first and final epochs always print; each report includes the epoch, elapsed time, ETA, objective, and active raw loss components. Regardless of that interval, every epoch is written to `metrics.jsonl` with raw components, weighted components, effective weights, learning rate, and timing. Thus `losses.png` shows the full per-epoch history. `training.checkpoint_every` separately controls restartable checkpoints. Final raw loss components, the best epoch, and the objective are saved in `final_training_metrics.json` and `summary.json`.

Each trainable log scale starts at `training.adaptive_loss_log_scale_init` (zero in the pilot), then is optimized jointly with the network using the configured optimizer and learning-rate schedule. A zero starting value makes every raw loss coefficient initially one in the code objective.


### Choose the training duration and PDE sampling budget

One epoch is one optimizer update for each selected method, using newly sampled interior points. There is no early stopping. `EPOCHS` is the update budget per method; it and the point counts must be positive integers. The PDE collocation points are sampled in the domain at each update. `BOUNDARY_POINTS` are sampled on each boundary for the standard PINN’s boundary penalty; exact-boundary methods do not use that penalty, and Nitsche has separate quadrature controls below.


In [ ]:
# Training settings; one epoch is one optimizer update per selected method.
EPOCHS = 6000
COLLOCATION_POINTS = 1024
BOUNDARY_POINTS = 256


### Choose an optimizer in PyTorch

The next cell constructs the optimizer directly. Replace `torch.optim.Adam(...)` with any optimizer whose `.step()` works with this training loop. You can set its arguments using PyTorch's usual keyword parameters.

In [ ]:
LEARNING_RATE = 0.001
OPTIMIZER_BETAS = (0.9, 0.999)


def make_optimizer(parameters):
    return torch.optim.Adam(parameters, lr=LEARNING_RATE, betas=OPTIMIZER_BETAS)


### Choose a learning-rate schedule in PyTorch

This is also a normal constructor call. Replace it with another `torch.optim.lr_scheduler` class if you want. The training loop calls `scheduler.step()` once after each optimizer update.

In [ ]:
SCHEDULE_GAMMA = 0.9999


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=SCHEDULE_GAMMA)


### Progress and restart checkpoints

Live reports show the epoch, elapsed time, ETA, objective, and active raw loss components. Every epoch is saved in `metrics.jsonl`; periodic checkpoints allow a stopped training run to resume.

In [ ]:
ADAPTIVE_LOSS_LOG_SCALE_INIT = 0.0
LOG_INTERVAL_SECONDS = 30
CHECKPOINT_EVERY = 100


## 9. Variational/Nitsche method: how its two trained losses are formed

Because the drift is a gradient, set $m(x)=e^{-V(x)/\varepsilon}$. The strong committor equation is equivalent to

$$
\mathcal Lq=0
\quad\Longleftrightarrow\quad
\nabla\cdot\left(\varepsilon m\nabla q\right)=0.
$$

For periodic Gaussian test functions $\phi_j$ centered at $c_j$, let $n_D$ be the outward normal of the exterior domain $D$, pointing into the removed sets:

$$
\phi_j(x)=\exp\!\left[
-\frac{4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)}{2\ell^2}
\right].
$$

The interior-plus-flux contribution is

$$
A_j=
\varepsilon\int_D m\nabla q_\theta\cdot\nabla\phi_j\,dx
-\varepsilon\int_{\partial D}m\,\partial_{n_D}q_\theta\,\phi_j\,ds.
$$

The symmetric consistency term and Nitsche stabilization moment are

$$
C_j=-\varepsilon\int_{\partial D}m(q_\theta-g)\,\partial_{n_D}\phi_j\,ds,
\qquad
P_j=\frac{\gamma\varepsilon}{h}
\int_{\partial D}m(q_\theta-g)\phi_j\,ds,
$$

where $g=0$ on $\partial A$ and $g=1$ on $\partial B$. Together they form the Nitsche boundary contribution $B_j=C_j+P_j$. The full weak residual is $R_j=A_j+B_j$.

The paper's VPINN loss split is

$$
\widehat J_{\rm pde}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}A_j^2,
\qquad
\widehat J_{\rm bc}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}B_j^2.
$$

These two losses are optimized with the common committor data term $\widehat J_{\rm data}$. In the training log they are named pde_loss, boundary_loss, and data_loss. Nitsche stabilization belongs inside the boundary component; it is not an extra loss on an exact-boundary ansatz or the standard PINN's pointwise boundary residual. The separate volume, flux, consistency, stabilization, and full-residual mean squares are diagnostics, not additional adaptive losses. This follows [Liao and Ming (2021)](https://doi.org/10.4208/cicp.OA-2020-0219) and [Berrone et al. (2023)](https://doi.org/10.1016/j.heliyon.2023.e18820).

**Check the equations in code:** the PDE and boundary geometry are in src/boundary_pinn/problems/committor/periodic_two_well.py; Monte Carlo and weak-form integrals are in paper_methods.py; ansatzes, active loss sets, adaptive scales, and epoch logs are in training.py; orchestration and saved error tables are in comparison.py.

### Set Nitsche integration controls (used only for Variational/Nitsche)

`NITSCHE_VOLUME_POINTS` controls interior quadrature, and `NITSCHE_BOUNDARY_POINTS` controls boundary quadrature. `NITSCHE_TEST_FUNCTIONS` is the number of Gaussian test functions; `TEST_FUNCTION_LENGTH` is their width. `STABILIZATION_GAMMA` controls the strength of the Nitsche stabilization term, and `BOUNDARY_SCALE_H` is the characteristic boundary length scale in that term. The test-function length and `h` are measured in the unit-torus coordinate scale. These settings affect only the Nitsche formulation.


In [ ]:
# Used only when "variational_nitsche" is in METHODS_TO_RUN.
NITSCHE_VOLUME_POINTS = 1024
NITSCHE_TEST_FUNCTIONS = 32
NITSCHE_BOUNDARY_POINTS = 512
TEST_FUNCTION_LENGTH = 0.12
STABILIZATION_GAMMA = 10.0
BOUNDARY_SCALE_H = 0.03


## 10. Choose reference accuracy and evaluation memory

Enter two increasing positive integer grid sizes, for example `[128, 256]`. The finer grid is used for solution-error estimates; the difference between grids diagnoses reference sensitivity, not a certified error bound. Larger grids use more time and memory. `EVALUATION_BATCH_SIZE` is the positive integer number of grid points evaluated together; lower it if evaluation runs out of memory.


In [ ]:
REFERENCE_GRID_SIZES = [128, 256]
EVALUATION_BATCH_SIZE = 16384


## 11. Check your code and prepare the run

Run the next cell after Setup and the CPU preflight pass in this runtime. Rerun it after editing the Python code and settings. It briefly checks that your sampler returns points in D, your model returns one finite value per point, and your PyTorch optimizer and scheduler can be constructed. This is a smoke check, not a training run. The cell then saves the scalar settings needed by the staged runner and prints the chosen model, sampler, optimizer, schedule, and training budget.

In [ ]:
# Serialization for the separate CPU and training stages. Edit the Python choices above.
import json
from pathlib import Path

problem_settings = {
    "a": POTENTIAL_A,
    "b": POTENTIAL_B,
    "epsilon": DIFFUSION_EPSILON,
    "source_center": list(SOURCE_CENTER),
    "target_center": list(TARGET_CENTER),
    "set_radius": COMMON_CHORDAL_RADIUS,
}
monte_carlo_settings = {
    "n_anchors": N_ANCHORS,
    "paths_per_anchor": PATHS_PER_ANCHOR,
    "dt": TIME_STEP,
    "max_steps": MAX_STEPS,
    "seed": MONTE_CARLO_SEED,
    "minimum_resolved_fraction": MINIMUM_RESOLVED_FRACTION,
    "progress_interval_fraction": PROGRESS_INTERVAL_FRACTION,
    "progress_interval_seconds": PROGRESS_INTERVAL_SECONDS,
}
nitsche_settings = {
    "volume_points": NITSCHE_VOLUME_POINTS,
    "test_functions": NITSCHE_TEST_FUNCTIONS,
    "boundary_points": NITSCHE_BOUNDARY_POINTS,
    "length_scale": TEST_FUNCTION_LENGTH,
    "gamma": STABILIZATION_GAMMA,
    "h": BOUNDARY_SCALE_H,
}
training_settings = {
    "epochs": EPOCHS,
    "collocation_points": COLLOCATION_POINTS,
    "boundary_points": BOUNDARY_POINTS,
    "learning_rate": LEARNING_RATE,
    "checkpoint_every": CHECKPOINT_EVERY,
    "log_interval_seconds": LOG_INTERVAL_SECONDS,
    "adaptive_loss_log_scale_init": ADAPTIVE_LOSS_LOG_SCALE_INIT,
    "monte_carlo": monte_carlo_settings,
    "nitsche": nitsche_settings,
}
RUN_CONFIG = {
    "seed": SEED,
    "device": DEVICE,
    "dtype": DTYPE,
    "methods": METHODS_TO_RUN,
    "problem": problem_settings,
    "training": training_settings,
    "reference_grid_sizes": REFERENCE_GRID_SIZES,
    "evaluation_batch_size": EVALUATION_BATCH_SIZE,
}
# Check the notebook's editable PyTorch code before CPU preparation.
NOTEBOOK_CODE_SMOKE_PASSED = False
try:
    from boundary_pinn.problems.committor.periodic_two_well import PeriodicTwoWellCommittor
except ModuleNotFoundError as error:
    if error.name and error.name.startswith("boundary_pinn"):
        raise RuntimeError(
            "The committor package is not installed in this Colab runtime. "
            "Run Setup (section 3) in this runtime, then rerun this cell. "
            "After changing the runtime type, Setup must be run again."
        ) from error
    raise

_probe_dtype = {"float32": torch.float32, "float64": torch.float64}[DTYPE]
_probe_problem = PeriodicTwoWellCommittor(
    a=POTENTIAL_A,
    b=POTENTIAL_B,
    epsilon=DIFFUSION_EPSILON,
    source_center=tuple(SOURCE_CENTER),
    target_center=tuple(TARGET_CENTER),
    set_radius=COMMON_CHORDAL_RADIUS,
)
_probe_points = sample_training_points(
    _probe_problem, 8, device="cpu", dtype=_probe_dtype
)
if not isinstance(_probe_points, torch.Tensor) or _probe_points.shape != (8, 2):
    raise ValueError("sample_training_points must return a tensor with shape [n, 2].")
if not bool(_probe_problem.domain_mask(_probe_points).all()):
    raise ValueError("sample_training_points returned points in A or B; return points in D.")
_probe_model = PeriodicCommittorNetwork().to(device="cpu", dtype=_probe_dtype)
_probe_values = _probe_model(_probe_points)
if _probe_values.shape != (8,) or not bool(torch.isfinite(_probe_values).all()):
    raise ValueError("The model must return one finite scalar per input point.")
_probe_optimizer = make_optimizer(_probe_model.parameters())
_probe_scheduler = make_scheduler(_probe_optimizer)
NOTEBOOK_CODE_SMOKE_PASSED = True
print("Model, sampler, optimizer, and scheduler smoke check passed.")

RUN_CONFIG_PATH = Path("/content/periodic_committor_selected_config.json")
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2) + "\n", encoding="utf-8")

print("Methods:", ", ".join(METHODS_TO_RUN))
print("Problem:", problem_settings)
print("Device / dtype / seed:", DEVICE, DTYPE, SEED)
print("Model architecture:")
print(_probe_model)
print("Interior sampling: sample_training_points() above")
print("Optimizer:", _probe_optimizer)
print("Learning-rate schedule:", _probe_scheduler)
print("Training settings:", training_settings)
print("Saved scalar settings:", RUN_CONFIG_PATH)
del _probe_points, _probe_values, _probe_model, _probe_optimizer, _probe_scheduler, _probe_problem


## 12. CPU preparation: reference solution and Monte Carlo labels

Run this after Setup, the CPU preflight, and the section 11 code check. This stage uses CPU whether Colab is running on CPU or GPU; keep the selected runtime attached. It does not train a network. It solves the finite-difference reference on `REFERENCE_GRID_SIZES` (default: `[128, 256]`), then estimates committor labels by simulating trajectories from the sampled anchors. It uses the selected problem, methods, and interior sampler. The network, optimizer, and scheduler are used later during training.

With the defaults above, Monte Carlo simulates 32 anchors × 256 paths = 8,192 trajectories, with `TIME_STEP = 0.001` and a cap of 100,000 steps per path. This is usually the slow part. Progress reports show the step cap, paths resolved in A and B, active paths, how many anchors meet the resolution threshold, the minimum and mean resolved fractions, and elapsed time. Reports appear at the configured fraction of the step cap or time interval, whichever comes first (defaults: 1,000 steps or 10 seconds).

Each anchor must meet `MINIMUM_RESOLVED_FRACTION` (default: 0.9). If it does not, preparation stops with an error; unresolved paths are not counted as hits of A. Reducing `N_ANCHORS` or `PATHS_PER_ANCHOR` makes the run faster but noisier. Lowering `MAX_STEPS` can leave more paths unresolved.

The dated Drive folder contains the saved configuration, code revision, reference, labels, progress metadata, and anchor plot. Keep its path for training. The model and other edited Python code remain in this notebook, so save this notebook copy with the run. If you change the preparation settings or sampler, rerun section 11 and create a new preparation folder.

In [ ]:
# Run CPU preparation and keep live progress in this cell.
from datetime import datetime
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive
from IPython.display import Image, display
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('Run Setup and pass the packaged CPU preflight in section 4 first.')
if not globals().get('NOTEBOOK_CODE_SMOKE_PASSED', False):
    raise RuntimeError('Run section 11 to smoke-check your model, sampler, optimizer, and scheduler first.')

torch.set_num_threads(2)
ensure_drive_mounted()
run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('CPU preparation folder (keep this path for training):', RUN_ROOT, flush=True)

run_periodic_comparison(
    RUN_CONFIG_PATH,
    Path(RUN_ROOT),
    prepare_only=True,
    interior_sampler=sample_training_points,
)
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
preflight_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('preparation_git_revision') != preflight_revision:
    raise RuntimeError('Preflight and prepared files came from different code revisions. Repeat setup and preparation from one branch revision.')
prep_record['cpu_preflight_passed'] = True
prep_record['cpu_preflight_git_revision'] = preflight_revision
prep_path.write_text(json.dumps(prep_record, indent=2) + '\n', encoding='utf-8')
print('CPU preparation is complete. No model training has run.', flush=True)
anchor_plot = Path(RUN_ROOT) / 'monte_carlo_anchors.png'
if anchor_plot.is_file():
    print('Monte Carlo anchor plot (inspect before training):', anchor_plot)
    display(Image(filename=str(anchor_plot)))


## 13. Training

The saved `DEVICE` setting chooses where training runs. If it is `"cuda"`, choose a GPU runtime before Setup and keep it connected; if it is `"cpu"`, choose a CPU runtime. CPU preparation has already completed without changing runtimes, so no restart or cell rerun is needed.

Run the next cell and paste the Drive folder printed during preparation. It checks the preparation marker and repository revision, loads the saved reference, labels, and configuration, then trains the selected methods with the model class, sampler, optimizer, and scheduler code above. It reports losses at the configured wall-clock interval and saves checkpoints in the same folder. It never repeats CPU preparation. The run records callback names; their source code remains in your saved notebook.

In [ ]:
# Train from the saved CPU preparation with the model, sampler, optimizer, and schedule above.
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

if not globals().get('NOTEBOOK_CODE_SMOKE_PASSED', False):
    raise RuntimeError('Rerun section 11 after defining or editing your model and callbacks.')

ensure_drive_mounted()
RUN_ROOT = input('Paste the CPU preparation folder path printed in the previous section: ').strip()
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise FileNotFoundError(f'No completed CPU preparation was found in {RUN_ROOT}.')
config_path = Path(RUN_ROOT, 'comparison_config.json')
if not config_path.is_file():
    raise FileNotFoundError(f'The prepared folder is missing its saved configuration: {RUN_ROOT}')
selected_config = json.loads(config_path.read_text(encoding='utf-8'))
training_device = str(selected_config['device'])
if training_device.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('This run selects CUDA, but this runtime has no GPU. Choose a GPU runtime, then rerun Setup and the notebook's code cells; the completed CPU preparation on Drive does not need to be repeated.')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
if prep_record.get('cpu_preflight_passed') is not True:
    raise RuntimeError('This folder was not prepared after a passing CPU preflight.')
current_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('cpu_preflight_git_revision') != current_revision:
    raise RuntimeError('This run folder belongs to a different code revision. Repeat Setup, preflight, and preparation from one branch revision.')

print(f'{training_device.upper()} training outputs will be saved in:', RUN_ROOT, flush=True)
run_periodic_comparison(
    config_path,
    Path(RUN_ROOT),
    train_prepared=True,
    model_factory=PeriodicCommittorNetwork,
    interior_sampler=sample_training_points,
    optimizer_factory=make_optimizer,
    scheduler_factory=make_scheduler,
)


## 14. Inspect the completed run

Run after training. In a fresh runtime, the cell asks for the Drive folder path. It reads saved files only and does not train. It displays the finite-difference grid check; absolute and relative $L^2$, $H^1$, $H^1$-seminorm, $H^2$, and $H^2$-seminorm errors; final raw loss components; and saved plots.

Errors use the finest configured finite-difference solution. Grid-to-grid differences estimate reference-resolution sensitivity, not an error bound. Figures are previews; arrays, metrics, and checkpoints are saved in the printed Drive folder.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    ensure_drive_mounted()
if 'RUN_ROOT' not in globals() or not Path(RUN_ROOT, 'summary.json').is_file():
    RUN_ROOT = input('Paste the completed Drive run folder path: ').strip()
if not Path(RUN_ROOT, 'summary.json').is_file():
    raise FileNotFoundError(f'No summary.json found in {RUN_ROOT}')

with open(f'{RUN_ROOT}/summary.json') as handle:
    summary = json.load(handle)

method_summaries = summary.get('methods', {})
accuracy_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
]
if method_summaries and all(
    all(field in metrics for field in accuracy_fields)
    for metrics in method_summaries.values()
):
    display(pd.DataFrame(method_summaries).T[accuracy_fields])
else:
    print(
        'This run summary predates the full Sobolev error report. '
        'Run the evaluation section to calculate absolute and relative L2, H1, and H2 errors '
        'from its saved checkpoints on CPU; no GPU retraining is needed.'
    )

print('Final raw training losses (not solution-error metrics):')
training_rows = {}
for method, metrics in method_summaries.items():
    training = metrics.get('final_training_metrics', {})
    raw_components = training.get('raw_loss_components', {})
    if not raw_components:
        metrics_path = Path(RUN_ROOT) / method / 'metrics.jsonl'
        if metrics_path.is_file():
            lines = [line for line in metrics_path.read_text().splitlines() if line.strip()]
            if lines:
                last = json.loads(lines[-1])
                raw_components = {
                    key: value for key, value in last.items()
                    if key.endswith('_loss') or key.endswith('_mse')
                }
                training = {
                    'final_training_objective': last.get('objective', last.get('loss')),
                    'raw_loss_components': raw_components,
                }
    training_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **raw_components,
    }
display(pd.DataFrame(training_rows).T)

print('Finite-difference grid comparison:')
print(json.dumps(summary.get('reference_convergence', {}), indent=2))
display(Image(filename=f'{RUN_ROOT}/losses.png'))
for method in method_summaries:
    figure = Path(RUN_ROOT) / f'committor_{method}.png'
    if figure.is_file():
        display(Image(filename=str(figure)))

## 15. Re-evaluate saved checkpoints on CPU

This separate cell reloads each method's `best.pt` checkpoint and recomputes errors against the saved finite-difference reference. It uses CPU only, does not retrain, and does not overwrite the original `summary.json`. It displays the Sobolev errors and final raw training-loss components and writes `saved_model_evaluation.json` to the same run folder.

In a fresh runtime, run the original model and initialization cells first, then paste the experiment folder path. Checkpoints contain weights, not your custom Python class; use the same model code that created them.

In [ ]:
# Re-evaluate saved best.pt checkpoints on CPU; this does not train.
from pathlib import Path
import json
from google.colab import drive
from IPython.display import display
import pandas as pd
from boundary_pinn.experiments.committor.periodic_two_well.comparison import evaluate_saved_models

if not Path('/content/drive/MyDrive').is_dir():
    ensure_drive_mounted()
if 'RUN_ROOT' not in globals() or not Path(RUN_ROOT, 'comparison_config.json').is_file():
    RUN_ROOT = input('Paste the completed experiment Drive folder path: ').strip()
if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError(f'No comparison_config.json found in {RUN_ROOT}')

evaluate_saved_models(Path(RUN_ROOT), model_factory=PeriodicCommittorNetwork)
report_path = Path(RUN_ROOT) / 'saved_model_evaluation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Training code revision:', report.get('training_git_revision') or 'not recorded for this older run')
print('FDM reference:', report['reference_file'])
display_fields = [
    'grid_l2_abs', 'grid_l2_rel', 'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel', 'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
    'source_boundary_rmse', 'target_boundary_rmse',
]
display(pd.DataFrame(report['methods']).T[display_fields])
print('Full saved report:', report_path)


## 16. Resume an interrupted run

Use this only when training stopped before all selected methods finished. A run saved with `device: cuda` needs a GPU runtime; a run saved with `device: cpu` uses CPU. If your current runtime already matches, keep it. If you change runtimes, Colab resets the live Python session, so rerun Setup and the notebook's code cells. Paste the saved Drive folder path. The runner resumes from each method's `last.pt`; it does not repeat preparation.

It checks the repository revision, saved objective version, and callback function names. It cannot compare the Python source inside notebook cells. Resume only with the exact model, sampler, optimizer, and schedule code used before; use a new dated folder if you changed them.

For a new experiment, return to the training section and use a new prepared folder.

In [ ]:
# Resume interrupted training only with the original model, sampler, optimizer, and schedule code.
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

if not globals().get('NOTEBOOK_CODE_SMOKE_PASSED', False):
    raise RuntimeError('Rerun section 11 after defining or editing your model and callbacks.')

ensure_drive_mounted()
RUN_ROOT = input('Paste the interrupted run folder path printed before training: ').strip()
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise FileNotFoundError('That folder has no completed CPU preparation.')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
config_path = Path(RUN_ROOT, 'comparison_config.json')
if not config_path.is_file():
    raise FileNotFoundError(f'The run folder is missing its saved configuration: {RUN_ROOT}')
selected_config = json.loads(config_path.read_text(encoding='utf-8'))
training_device = str(selected_config['device'])
if training_device.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('This saved run requires a GPU runtime. Choose GPU, then rerun Setup and the notebook's code cells; do not repeat the completed CPU preparation.')
current_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('cpu_preflight_passed') is not True or prep_record.get('cpu_preflight_git_revision') != current_revision:
    raise RuntimeError('This run folder does not match the passed CPU preflight and installed code revision.')

run_periodic_comparison(
    config_path,
    Path(RUN_ROOT),
    train_prepared=True,
    resume=True,
    model_factory=PeriodicCommittorNetwork,
    interior_sampler=sample_training_points,
    optimizer_factory=make_optimizer,
    scheduler_factory=make_scheduler,
)
print(f'The interrupted {training_device.upper()} run has resumed and completed.')


## References

- Y. Liao and P. Ming, “Deep Nitsche Method: Deep Ritz Method with Essential Boundary Conditions,” *Communications in Computational Physics* 29(5) (2021), 1365–1384. [doi:10.4208/cicp.OA-2020-0219](https://doi.org/10.4208/cicp.OA-2020-0219).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, “Physics-informed neural networks,” *Journal of Computational Physics* 378 (2019), 686–707. [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045).
- N. Sukumar and A. Srivastava, “Exact imposition of boundary conditions with distance functions in physics-informed deep neural networks,” *Computer Methods in Applied Mechanics and Engineering* 389 (2022), 114333. [doi:10.1016/j.cma.2021.114333](https://doi.org/10.1016/j.cma.2021.114333).
- S. Berrone, C. Canuto, M. Pintore, and N. Sukumar, “Enforcing Dirichlet boundary conditions in physics-informed neural networks and variational physics-informed neural networks,” *Heliyon* 9(8) (2023), e18820. [doi:10.1016/j.heliyon.2023.e18820](https://doi.org/10.1016/j.heliyon.2023.e18820).
- A. Kendall, Y. Gal, and R. Cipolla, “Multi-task learning using uncertainty to weigh losses for scene geometry and semantics,” *CVPR* (2018). [paper](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).
- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, “Boundary-Adapted PINNs for Elliptic Dirichlet Problems,” [arXiv:2607.19167](https://arxiv.org/abs/2607.19167) (2026).